In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
# Model parameters
DOCS_DIR = "../docs"
PERSIST_DIR = "../storage"
LLM_MODEL = "granite3.1-dense:8b"
EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"

In [5]:
import os
from langchain_chroma import Chroma
from langchain.document_loaders import DirectoryLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [14]:
## VECTOR STORE
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

# Check if the vector store already exists
if os.path.exists(PERSIST_DIR):
    print("Loading existing vector store...")
    vector_store = Chroma(
        persist_directory=PERSIST_DIR, embedding_function=embeddings
    )
else:
    print("Creating new vector store...")
    os.mkdir(PERSIST_DIR)
    # Process documents for RAG
    loader = DirectoryLoader(DOCS_DIR)
    docs = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000, chunk_overlap=200
    )
    all_splits = text_splitter.split_documents(docs)

    # Create a new vector store and add documents
    vector_store = Chroma.from_documents(
        documents=all_splits,
        embedding=embeddings,
        persist_directory=PERSIST_DIR,
    )
    print("Vector store created and documents indexed.")
print("Vector store loaded")

Loading existing vector store...
Vector store loaded


In [15]:
print(vector_store._collection.count())

25


In [16]:
from langchain_ollama import ChatOllama

In [17]:
# Load LLM + embedding model
llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0
)

In [18]:
from langchain import hub
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [19]:
prompt = hub.pull("rlm/rag-prompt")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {
        "context": vector_store.as_retriever() | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [20]:
query = "Where did Tim Forrer go to university?"
response = qa_chain.invoke(query)
print(response)

Tim Forrer is currently pursuing his Physics PhD at the University of Tokyo, Japan. Prior to this, he completed a Natural Sciences MSci (1st Class Honors) from Durham University in the UK.
